## Week 1

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [ ]:
df = pd.read_csv(r"C:/Users/sakin_xyc6lmd/OneDrive/Desktop/Infotact internship projects/SaaS-E-Commerce-Cohort-Retention-CLTV-Analysis-Infotact-Group-5---Project-1-/OnlineRetail.csv", encoding='latin1')

In [ ]:
# first 20 rows

df.head(20)

In [ ]:
# how many rows and columns

df.shape

In [ ]:
# column types and null counts

df.info()

In [ ]:
# how many missing values per column

df.isnull().sum()

In [ ]:
# removing null values

df = df.dropna(subset=['CustomerID'])
print("After removing all null CustomerIDs:", df.shape)

In [ ]:
#Cancelled Orders

cancelled = df[df['InvoiceNo'].astype(str).str.startswith('C')]
print(cancelled.shape)
print(cancelled.head(10)) 

In [ ]:
#Removing cancelled orders

df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
print("After removing cancellations:", df.shape)

In [ ]:
#Removing negative and zero quantities 

df = df[df['Quantity'] > 0]
print("After removing zero and negative quantities:", df.shape)

In [ ]:
#Removing zero and negative prices

df = df[df['UnitPrice'] > 0]
print("After removing zero and negative prices:", df.shape)

In [ ]:
#Resetting Index

df = df.reset_index(drop=True)
print("Final Cleaned Dataset:", df.shape)
print(df.head(20))

In [ ]:
print(df.dtypes)

In [ ]:
# converting customerid from float to int

df['CustomerID'] = df['CustomerID'].astype(int)
print(df['CustomerID'].dtype)
print(df['CustomerID'].head())

In [ ]:
# convertng InvoiceDate to datetime dtype

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(df['InvoiceDate'].dtype)
print(df['InvoiceDate'].head()) 

In [ ]:
# null check

print(df.isnull().sum())

In [ ]:
print(df.shape)
print(df.dtypes)
print(df.head())

In [ ]:
# Extracting InvoiceMonth from InvoiceDate

df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M')
df.InvoiceMonth.head()

In [ ]:
#Each Customer's First Purchase Month

cohort_month = df.groupby('CustomerID')['InvoiceMonth'].min().reset_index()
cohort_month.columns = ['CustomerID', 'CohortMonth']
print(cohort_month.head(20))
print(cohort_month.shape)

In [ ]:
#Merging CohortMonth Back Into Main Dataframe

df = df.merge(cohort_month, on='CustomerID', how='left')
print(df.head(20))
print(df.shape)

In [ ]:
#checking

df[['CustomerID', 'InvoiceMonth', 'CohortMonth']].head(50)

In [ ]:
print(df['CohortMonth'].nunique())
print(df['CohortMonth'].unique())

In [ ]:
#TotalPrice Column

df['TotalPrice'] = df['Quantity'] * df['UnitPrice']
print(df['TotalPrice'].head(10))

In [ ]:
print(df['TotalPrice'].describe())

In [ ]:
#top 20 customers

top_customers = df.groupby('CustomerID')['TotalPrice'].sum().sort_values(ascending=False).head(20)
print(top_customers)

In [ ]:
#revenue by country

revenue_by_country = df.groupby('Country')['TotalPrice'].sum().sort_values(ascending=False).head(20)
print(revenue_by_country)

In [ ]:
#top 20 products

top_products = df.groupby('Description')['Quantity'].sum().sort_values(ascending=False).head(20)
print(top_products)

In [ ]:
#monthly revenue

monthly_revenue = df.groupby('InvoiceMonth')['TotalPrice'].sum().sort_values(ascending=False)
print(monthly_revenue)

In [ ]:
# Reload raw data to examine cancellations separately

df_raw = pd.read_csv(r"C:/Users/sakin_xyc6lmd/OneDrive/Desktop/Infotact internship projects/SaaS-E-Commerce-Cohort-Retention-CLTV-Analysis-Infotact-Group-5---Project-1-/OnlineRetail.csv", encoding='latin1')
cancelled = df_raw[df_raw['InvoiceNo'].astype(str).str.startswith('C')]
print("Total cancelled orders:", cancelled.shape)
print(cancelled.head(20)) 

In [ ]:
#which customer cancelled most

cancelled_by_customer = cancelled.groupby('CustomerID')['InvoiceNo'].count().sort_values(ascending=False).head(20)
print(cancelled_by_customer)

In [ ]:
#cancelled by country

cancelled_by_country = cancelled.groupby('Country')['InvoiceNo'].count().sort_values(ascending=False).head(10)
print(cancelled_by_country)

In [ ]:
print("Final cleaned dataset shape:", df.shape)
print(df.dtypes)
print(df.isnull().sum())

## Week 1 — Data Cleaning & EDA
**Dataset:** UCI Online Retail Dataset — 541,909 raw transactions

**Cleaning done:** Removed missing CustomerIDs, cancelled orders, 
negative quantities/prices. Fixed CustomerID to int, InvoiceDate to datetime.

**New columns:** InvoiceMonth, CohortMonth (first purchase month per 
customer), TotalPrice (Quantity x UnitPrice)

**Cohort definition:** A cohort = customers who made their first 
purchase in the same month. Cancellations ≠ churn — churn means not 
returning in later months.

**Final shape:** 397,884 rows x 10 columns

## WEEK 2 

In [ ]:
# Calculate CohortIndex

invoice_year = df['InvoiceMonth'].dt.year
invoice_month = df['InvoiceMonth'].dt.month
cohort_year = df['CohortMonth'].dt.year
cohort_month = df['CohortMonth'].dt.month

In [ ]:
#how many months have passed between a customer's first purchase (cohort month) and each individual transaction
df['CohortIndex'] = (invoice_year - cohort_year) * 12 + (invoice_month - cohort_month) + 1

In [ ]:
# Build cohort matrix
cohort_data = df.groupby(['CohortMonth', 'CohortIndex'])['CustomerID'].nunique().reset_index()
cohort_matrix = cohort_data.pivot(index='CohortMonth', columns='CohortIndex', values='CustomerID')

In [ ]:
# Convert to percentage retention
cohort_sizes = cohort_matrix[1]
retention_matrix = cohort_matrix.divide(cohort_sizes, axis=0) * 100

In [ ]:
print(retention_matrix.round(1))


## Week 2 — Cohort Retention Matrix
**Objective:** Track how many customers from each cohort keep 
purchasing in the months after joining.

**Steps:** Calculated CohortIndex (months since first purchase) → 
built raw cohort matrix (groupby + pivot) → converted to % retention 
by dividing by Month 1 size.

**Key findings:**
- Retention drops to ~15-25% by Month 2 across most cohorts
- Dec 2010 cohort retains best long-term (~26.6% at Month 13)
- After Month 2, retention stabilises around 15-35%
- Recommendation: target re-engagement in the Month 1→2 window

## Week 3

In [ ]:
# Group by customer: total revenue and total number of orders
customer_summary = df.groupby('CustomerID').agg(TotalRevenue=('TotalPrice', 'sum'),TotalOrders=('InvoiceNo', 'nunique')).reset_index()

# Average Order Value = Total Revenue / Total Orders
customer_summary['AOV'] = customer_summary['TotalRevenue'] / customer_summary['TotalOrders']

print(customer_summary.head(10))

In [ ]:
# Average purchase frequency across ALL customers
avg_purchase_frequency = customer_summary['TotalOrders'].mean()

print("Average Purchase Frequency:", round(avg_purchase_frequency, 2))

In [ ]:
customer_summary['CLTV'] = customer_summary['AOV'] * customer_summary['TotalOrders']

# Sort to see highest value customers first
print(customer_summary.sort_values('CLTV', ascending=False).head(10))

In [ ]:
customer_summary['ValueSegment'] = pd.qcut(customer_summary['CLTV'], q=3, labels=['Low', 'Medium', 'High'])

print(customer_summary.groupby('ValueSegment', observed=True)['CLTV'].agg(['mean', 'count']))


In [ ]:
total_cltv_by_segment = customer_summary.groupby('ValueSegment', observed=True)['CLTV'].sum()
print(total_cltv_by_segment)

print("\nPercentage of total revenue by segment:")
print((total_cltv_by_segment / total_cltv_by_segment.sum() * 100).round(1))

In [ ]:
import matplotlib.pyplot as plt

segment_revenue = customer_summary.groupby('ValueSegment', observed=True)['CLTV'].sum()

segment_revenue.plot(kind='bar', color=['#C8EEE1', '#5DCAA5', '#0F6E56'])
plt.title('Total CLTV by Customer Segment')
plt.ylabel('Total CLTV (₹)')
plt.xlabel('Segment')
plt.show()

## Week 3 — CLTV Calculation (Key Finding)
- Customers segmented into Low/Medium/High value tiers (1446 each)
- High segment average CLTV: ₹5229.61 vs Low segment: ₹220.60
- High segment contributes 84.9% of total revenue despite being 
  only 33% of customer base
- Strong Pareto pattern — business is highly dependent on top-tier customers
- Recommendation: prioritize retention for High segment, explore 
  upsell paths for Medium segment